# Poker Prediction Algorithm - Quick Start

This notebook demonstrates the end-to-end pipeline for poker prediction.

## 1. Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.preprocess import PokerDataPreprocessor
from src.features.engineering import PokerFeatureEngineer
from src.models.train_ml import PokerMLTrainer

%matplotlib inline
sns.set_style('whitegrid')

## 2. Load and Explore Data

In [ ]:
# Load raw data
df_train = pd.read_csv('../data/raw/pokerbench/train.csv')
print(f"Training samples: {len(df_train)}")
print(f"\nColumns: {list(df_train.columns)}")
df_train.head()

In [ ]:
# Explore target distribution
if 'correct_decision' in df_train.columns:
    target_counts = df_train['correct_decision'].value_counts()
    print("Decision Distribution:")
    print(target_counts)
    
    plt.figure(figsize=(10, 6))
    target_counts.plot(kind='bar')
    plt.title('Distribution of Optimal Decisions')
    plt.xlabel('Decision')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 3. Preprocess Data

In [ ]:
# Initialize preprocessor
preprocessor = PokerDataPreprocessor(raw_data_dir='../data/raw/pokerbench')

# Load and preprocess
df_train_processed = preprocessor.extract_preflop_features(df_train)
print(f"Processed shape: {df_train_processed.shape}")
df_train_processed.head()

## 4. Feature Engineering

In [ ]:
# Initialize feature engineer
engineer = PokerFeatureEngineer()

# Engineer features
df_features = engineer.engineer_features(df_train_processed)
print(f"Feature shape: {df_features.shape}")
print(f"\nSample features:")
df_features.head()

In [ ]:
# Analyze engineered features
numeric_features = df_features.select_dtypes(include=[np.number]).columns
print(f"Number of numeric features: {len(numeric_features)}")

# Check for missing values
missing = df_features[numeric_features].isnull().sum()
print(f"\nMissing values:\n{missing[missing > 0]}")

## 5. Train Model

In [ ]:
from sklearn.model_selection import train_test_split

# Initialize trainer
trainer = PokerMLTrainer(model_type='xgboost')

# Prepare features
X, y = trainer.prepare_features(df_features)

# Split data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train)}")
print(f"Val size: {len(X_val)}")

In [ ]:
# Train model
results = trainer.train(X_train, y_train, X_val, y_val)
print(f"\nTraining Results:")
for key, value in results.items():
    print(f"  {key}: {value}")

## 6. Feature Importance

In [ ]:
# Get feature importance
importance_df = trainer.get_feature_importance(top_n=20)

# Plot
plt.figure(figsize=(12, 8))
plt.barh(range(len(importance_df)), importance_df['importance'])
plt.yticks(range(len(importance_df)), importance_df['feature'])
plt.xlabel('Importance')
plt.title('Top 20 Most Important Features')
plt.tight_layout()
plt.show()

## 7. Evaluate on Test Set

In [ ]:
# Load test data
df_test = pd.read_csv('../data/raw/pokerbench/test.csv')

# Preprocess
df_test_processed = preprocessor.extract_preflop_features(df_test)

# Engineer features
df_test_features = engineer.engineer_features(df_test_processed)

# Prepare test set
X_test, y_test = trainer.prepare_features(df_test_features)

print(f"Test size: {len(X_test)}")

In [ ]:
# Evaluate
test_results = trainer.evaluate(X_test, y_test)

print(f"\nTest Accuracy: {test_results['accuracy']:.4f}")

## 8. Make Predictions

In [ ]:
# Example prediction
sample_idx = 0
sample_features = X_test.iloc[sample_idx:sample_idx+1]

# Predict
pred_encoded = trainer.model.predict(sample_features)
pred = trainer.label_encoder.inverse_transform(pred_encoded)[0]

# Get probabilities
pred_proba = trainer.model.predict_proba(sample_features)[0]

print(f"Sample Prediction:")
print(f"  Predicted: {pred}")
print(f"  Actual: {y_test.iloc[sample_idx]}")
print(f"\nProbabilities:")
for label, prob in zip(trainer.label_encoder.classes_, pred_proba):
    print(f"  {label}: {prob:.4f}")

## 9. Save Model

In [ ]:
# Save trained model
trainer.save_model('../data/models/xgboost_quickstart.pkl')
print("Model saved successfully!")

## Next Steps

1. Try different model types (LightGBM, Random Forest)
2. Experiment with feature engineering
3. Tune hyperparameters
4. Train on full dataset
5. Try neural network models
6. Fine-tune an LLM